In [1]:
# Google Colab에서 한국어 분석에 필요한 Java와 KoNLPy 설치
# --quiet 옵션: 설치 로그를 간결하게 출력
!apt-get install default-jdk -y
!pip install konlpy scikit-learn --quiet
print("✅ 완료!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core default-jdk-headless default-jre default-jre-headless
  fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk openjdk-11-jdk-headless openjdk-11-jre
  openjdk-11-jre-headless session-migration x11-utils
Suggested packages:
  libxt-doc openjdk-11-demo openjdk-11-source visualvm libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core default-jdk default-jdk-headless default-jre
  default-jre-headless fonts-dejavu-core fonts-dejavu-extra
  gsettings-desktop-schemas libatk-bridge2.0-0 libatk-wrapper-java
  libatk-wrapper-java-jn

---
## 🏆 도전 과제 — 뉴스 기사 분류

**목표:** 같은 주제 기사끼리 유사도가 높게 나오면 성공! 🎯

| 기사 | 주제 |
|------|------|
| 기사1, 기사2 | 스마트폰 |
| 기사3, 기사4 | 스포츠 |
| 기사5, 기사6 | AI |

In [2]:
import re          # 정규표현식: 텍스트 정제에 사용
import math        # log() 함수: IDF 계산에 사용

from konlpy.tag import Okt                                    # 한국어 형태소 분석기
from sklearn.feature_extraction.text import TfidfVectorizer   # TF-IDF 자동 계산
from sklearn.metrics.pairwise import cosine_similarity        # 문서 유사도 계산

okt = Okt()   # 형태소 분석기 객체 생성 (최초 1회, 시간이 걸릴 수 있음)
print("✅ 준비 완료!")

✅ 준비 완료!


In [20]:

# ── 전처리 파이프라인 함수 정의 ─────────────────
# 모든 단계를 하나로 묶은 함수입니다

# 분석에 불필요한 한국어 단어 목록
불용어_목록 = [
    "것", "수", "나", "저", "제", "그", "이", "때",
    "등", "좀", "잘", "더", "한", "안", "못", "또"
]

def 전처리(텍스트):
    """
    입력: 원본 텍스트 (문자열)
    출력: 전처리된 핵심 단어 리스트
    """
    # 1단계: 정제 — 노이즈 제거
    텍스트 = re.sub(r'http\S+', '', 텍스트)               # URL 제거
    텍스트 = re.sub(r'<[^>]+>', '', 텍스트)               # HTML 태그 제거
    텍스트 = re.sub(r'[^가-힣a-zA-Z0-9 ]', '', 텍스트)    # 특수문자 제거
    텍스트 = re.sub(r'\s+', ' ', 텍스트).strip()           # 공백 정리

    # 2단계: 정규화 — 반복 문자 압축
    텍스트 = re.sub(r'(.)\1{2,}', r'\1\1', 텍스트)

    # 3단계: 형태소 분석 — 명사 추출
    토큰들 = okt.nouns(텍스트)

    # 4단계: 불용어 제거 + 2글자 이상만 유지
    결과 = [
        단어 for 단어 in 토큰들
        if 단어 not in 불용어_목록  # 불용어 목록에 없는 것만
        and len(단어) >= 2          # 2글자 이상인 것만
    ]

    return 결과

print("✅ 파이프라인 함수 준비 완료!")

✅ 파이프라인 함수 준비 완료!


In [15]:
# 뉴스 기사 6개
기사1 = "삼성전자가 신형 갤럭시 스마트폰을 출시했습니다. AI 카메라 기능과 배터리 성능이 향상됐습니다."
기사2 = "애플이 아이폰 신제품을 발표했습니다. 최신 칩셋과 카메라를 개선했으며 배터리 수명도 늘었습니다."
기사3 = "국내 프로야구 한국시리즈가 개막했습니다. 투수진과 타선의 조화가 우승의 핵심 요소입니다."
기사4 = "국가대표 축구팀이 월드컵 예선에서 승리했습니다. 공격수의 결승골이 팀을 이끌었습니다."
기사5 = "정부가 인공지능 산업 육성을 위한 투자 계획을 발표했습니다. AI 반도체 개발도 지원합니다."
기사6 = "국내 스타트업이 AI 기반 의료 진단 서비스를 출시했습니다. 인공지능으로 암 조기 발견율을 높입니다."

In [21]:
# STEP 1: 전처리 — 6개 기사 각각 명사 추출
처리_기사1 = 전처리(기사1)
처리_기사2 = 전처리(기사2)
처리_기사3 = 전처리(기사3)
처리_기사4 = 전처리(기사4)
처리_기사5 = 전처리(기사5)
처리_기사6 = 전처리(기사6)

print("전처리 결과 (명사만 추출):")
print(f"  기사1: {처리_기사1}")
print(f"  기사2: {처리_기사2}")
print(f"  기사3: {처리_기사3}")
print(f"  기사4: {처리_기사4}")
print(f"  기사5: {처리_기사5}")
print(f"  기사6: {처리_기사6}")

전처리 결과 (명사만 추출):
  기사1: ['전자', '신형', '갤럭시', '스마트폰', '출시', '카메라', '기능', '배터리', '성능', '향상']
  기사2: ['애플', '아이폰', '신제품', '발표', '최신', '칩셋', '카메라', '개선', '배터리', '수명']
  기사3: ['국내', '프로야구', '한국', '시리즈', '개막', '투수', '진과', '선의', '조화', '우승', '핵심', '요소']
  기사4: ['국가대표', '축구팀', '월드컵', '예선', '승리', '공격수', '결승골']
  기사5: ['정부', '인공', '지능', '산업', '육성', '투자', '계획', '발표', '반도체', '개발', '지원']
  기사6: ['국내', '스타트업', '기반', '의료', '진단', '서비스', '출시', '인공', '지능', '조기', '발견']


In [22]:
# STEP 2: TF-IDF 행렬 생성
# 전처리 결과(리스트)를 ' '.join()으로 문자열로 변환 후 vectorizer에 전달
기사_vec = TfidfVectorizer()
기사_tfidf = 기사_vec.fit_transform([
    ' '.join(처리_기사1), ' '.join(처리_기사2),
    ' '.join(처리_기사3), ' '.join(처리_기사4),
    ' '.join(처리_기사5), ' '.join(처리_기사6)
])
기사_어휘 = 기사_vec.get_feature_names_out()
기사_배열 = 기사_tfidf.toarray()

print(f"전체 어휘: {len(기사_어휘)}개")
print(f"행렬 크기: {기사_tfidf.shape}  ← (기사 6개 × 단어 수)")

전체 어휘: 54개
행렬 크기: (6, 54)  ← (기사 6개 × 단어 수)


In [23]:
# STEP 3: 기사별 핵심어 (.argmax()로 TF-IDF 최고 단어 추출)
print("기사별 핵심어:")
print(f"  기사1 [스마트폰]: '{기사_어휘[기사_배열[0].argmax()]}'  ({기사_배열[0].max():.3f})")
print(f"  기사2 [스마트폰]: '{기사_어휘[기사_배열[1].argmax()]}'  ({기사_배열[1].max():.3f})")
print(f"  기사3 [스포츠]:   '{기사_어휘[기사_배열[2].argmax()]}'  ({기사_배열[2].max():.3f})")
print(f"  기사4 [스포츠]:   '{기사_어휘[기사_배열[3].argmax()]}'  ({기사_배열[3].max():.3f})")
print(f"  기사5 [AI]:       '{기사_어휘[기사_배열[4].argmax()]}'  ({기사_배열[4].max():.3f})")
print(f"  기사6 [AI]:       '{기사_어휘[기사_배열[5].argmax()]}'  ({기사_배열[5].max():.3f})")

기사별 핵심어:
  기사1 [스마트폰]: '갤럭시'  (0.333)
  기사2 [스마트폰]: '개선'  (0.333)
  기사3 [스포츠]:   '개막'  (0.293)
  기사4 [스포츠]:   '결승골'  (0.378)
  기사5 [AI]:       '개발'  (0.316)
  기사6 [AI]:       '기반'  (0.321)


In [24]:
# STEP 4: 유사도 행렬 — 같은 주제 기사끼리 높게 나오면 성공!
기사_유사도 = cosine_similarity(기사_tfidf)

print("유사도 확인:")
print()
# 같은 주제 쌍: 높아야 함
print(f"  [같은 주제] 스마트폰: 기사1 ↔ 기사2 = {기사_유사도[0][1]:.3f}  {'✅ 높음!' if 기사_유사도[0][1] > 0.1 else '🤔 낮음'}")
print(f"  [같은 주제] 스포츠:   기사3 ↔ 기사4 = {기사_유사도[2][3]:.3f}  {'✅ 높음!' if 기사_유사도[2][3] > 0.1 else '🤔 낮음'}")
print(f"  [같은 주제] AI:       기사5 ↔ 기사6 = {기사_유사도[4][5]:.3f}  {'✅ 높음!' if 기사_유사도[4][5] > 0.1 else '🤔 낮음'}")
print()
# 다른 주제 쌍: 낮아야 함
print(f"  [다른 주제] 기사1(스마트폰) ↔ 기사3(스포츠) = {기사_유사도[0][2]:.3f}  {'✅ 낮음!' if 기사_유사도[0][2] < 0.1 else '🤔 높음'}")
print(f"  [다른 주제] 기사1(스마트폰) ↔ 기사5(AI)     = {기사_유사도[0][4]:.3f}  {'✅ 낮음!' if 기사_유사도[0][4] < 0.1 else '🤔 높음'}")
print()
print("같은 주제끼리 유사도 높고, 다른 주제끼리 낮으면 성공! 🎯")

유사도 확인:

  [같은 주제] 스마트폰: 기사1 ↔ 기사2 = 0.149  ✅ 높음!
  [같은 주제] 스포츠:   기사3 ↔ 기사4 = 0.000  🤔 낮음
  [같은 주제] AI:       기사5 ↔ 기사6 = 0.137  ✅ 높음!

  [다른 주제] 기사1(스마트폰) ↔ 기사3(스포츠) = 0.000  ✅ 낮음!
  [다른 주제] 기사1(스마트폰) ↔ 기사5(AI)     = 0.000  ✅ 낮음!

같은 주제끼리 유사도 높고, 다른 주제끼리 낮으면 성공! 🎯
